# 1. Presto C++ - CPU

A two-container Presto cluster from Docker Hub: a Java **coordinator** and a C++ **[native](https://github.com/facebookincubator/velox) worker**.

## 1.1 Minimal Required Host Files

Dirs are mounted into the containers:
 - `metastore/` = metadata
 - `warehouse/` = table data
 - `etc_CPU_coordinator/` = Presto coordinator config
 - `etc_CPU_coordinator/catalog` = hold Hive properties
 - `etc_CPU_worker/` = Presto worker config
  
Required Configs:

`etc_CPU_coordinator/`
 - config.properties
 - node.properties
 - jvm.config
 - `catalog/`hive.properties

`etc_CPU_worker/`
 - config.properties
 - node.properties
 - catalog/hive.properties


In [1]:
!mkdir -p metastore warehouse

In [ ]:
import os
%env PRESTO_UID={str(os.getuid())}
%env PRESTO_GID={str(os.getgid())}

## 1.2 Start Presto
Containers:
 | container | function | count | impl |
 |---|---|---|---|
 | prestodb/presto:latest| presto coordinator | 1 | Java |
 | prestodb/presto-native:latest| presto native worker | 1+ | C++ |

In [2]:
!docker compose -f docker-compose-presto-native.yaml up --pull always -d

# !docker compose -f docker-compose-presto-native.yaml down

In [3]:
!docker ps -a

CONTAINER ID   IMAGE                           COMMAND                CREATED          STATUS                    PORTS                                         NAMES
57a519eccf63   prestodb/presto-native:latest   "/opt/entrypoint.sh"   42 seconds ago   Up 41 seconds (healthy)   0.0.0.0:8082->8082/tcp, [::]:8082->8082/tcp   worker-cpu
2bc7095190ab   prestodb/presto:latest          "/opt/entrypoint.sh"   42 seconds ago   Up 41 seconds (healthy)   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp   coordinator-cpu


## 1.3 Smoke Test

In [4]:
import prestodb

conn = prestodb.dbapi.connect(host="localhost", 
                              port=8080, 
                              user="nb", 
                              catalog="hive", 
                              schema="demo")

cur = conn.cursor()

In [5]:
!mkdir -p warehouse/demo/fruit

cur.execute("CREATE SCHEMA IF NOT EXISTS hive.demo")

cur.execute("""
          CREATE TABLE IF NOT EXISTS 
               hive.demo.fruit (name VARCHAR, qty INTEGER) 
          WITH (format='PARQUET', external_location='file:/home/warehouse/demo/fruit')
          """)


In [6]:
cur.execute("""
          INSERT INTO 
              hive.demo.fruit 
          VALUES 
              ('apple', 3), 
              ('banana', 5)
          """)

In [7]:
cur.execute("SELECT * FROM hive.demo.fruit")
cur.fetchall()

[['apple', 3], ['banana', 5]]

In [8]:
cur.execute(""" DROP TABLE IF EXISTS  hive.demo.fruit """)
cur.execute(""" DROP SCHEMA IF EXISTS  hive.demo """)
cur.execute("SHOW SCHEMAS FROM hive")
cur.fetchall()

[['information_schema']]

In [9]:
!rm -rf warehouse/demo

In [10]:
cur.close()